<!-- DRAFT: placeholder prose; the author will rewrite this. -->
# 03 · Keeping answers current
A policy changes. In one stream transaction we insert its new version (with its embedding and parsed edges), close the old one, and add a `supersedes` edge. The next query already answers from the new version: nothing is re-indexed. This runs in a copy of the database (`northwind_kb_demo`).

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, '..'); sys.path.insert(0, '../scripts')
import fresh_demo
fresh_demo.main()

Building northwind_kb_demo from data/ (a copy; the eval database is not touched)


W1118 17:23:33.073000 13486 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

kb_view commitIntervalMsec = 1000 (BM25 queries below use waitForSync, so they see a commit immediately)

Question (asked 2026-11-01): "How will you pay me for my car, and how long does it take?"

### 1. Before the policy change
| method | #1 | getting_paid versions in the top 5 |
|---|---|---|
| M1 BM25 | `how_selling_works__2026` current | none |
| M2 Vector | `getting_paid__2022` **STALE** | #1 `getting_paid__2022` **STALE** |
| M3 Vector + filter | `how_selling_works__2026` current | #4 `getting_paid__2026` current |
| M4 Graph: structural | `how_selling_works__2026` current | #4 `getting_paid__2026` current |


| M5 Graph: semantic | `how_selling_works__2026` current | #4 `getting_paid__2026` current |
| M6 Graph: both layers | `how_selling_works__2026` current | #4 `getting_paid__2026` current |

### 2. The policy changes: one stream transaction
AQL> INSERT @doc INTO articles RETURN NEW._key
bind: {'doc': {'_key': 'getting_paid__2026-11', 'doc_id': 'getting_paid', 'version': '2026-11', 'kind': 'rewritten', 'effective_from': '2026-11-01', 'effective_to': None, 'states': [], 'side': 'sell', 'section': 'loans', 'title': 'When and how you get paid', 'text': 'Once the paperwork is signed, we pay you by bank draft or by instant transfer to your bank account, which arrives within minutes on any day of the week. Dire...', 'codes': [], 'embedding': '<384 floats>'}}
inserted: ['getting_paid__2026-11']
AQL> UPDATE @key WITH { effective_to: @closed } IN articles RETURN { key: NEW._key, effective_to: NEW.effective_to }
bind: {'key': 'getting_paid__2026', 'closed': '2026-10-31'}
closed:   [{'key': 'gettin

| M5 Graph: semantic | `how_selling_works__2026` current | #5 `getting_paid__2026-11` current |
| M6 Graph: both layers | `how_selling_works__2026` current | #5 `getting_paid__2026-11` current |

(all four queries after the commit: 147 ms)

### 4. What the current answer replaced: walk `supersedes`
AQL> FOR v, e IN 0..5 OUTBOUND @start supersedes
       RETURN { version: v._key, in_effect: CONCAT(v.effective_from, " to ", v.effective_to || "now"), replaced_it_on: e.on }
bind: {'start': 'articles/getting_paid__2026-11'}
   {'version': 'getting_paid__2026-11', 'in_effect': '2026-11-01 to now', 'replaced_it_on': None}
   {'version': 'getting_paid__2026', 'in_effect': '2026-02-01 to 2026-10-31', 'replaced_it_on': '2026-11-01'}
   {'version': 'getting_paid__2022', 'in_effect': '2022-01-01 to 2026-01-31', 'replaced_it_on': '2026-02-01'}


0